# 03 - DistilBERT fine-tuning

This notebook makes the Transformer inputs concrete, then calls the reusable training pipeline. A CUDA GPU is recommended. Running the final cell downloads `distilbert-base-uncased`, fine-tunes it, evaluates the untouched test split, and saves the best model locally.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
sys.path.insert(0, str(PROJECT_ROOT))
from transformers import AutoTokenizer
from src.config import settings
from src.data_loader import prepare_data
splits = prepare_data()
tokenizer = AutoTokenizer.from_pretrained(settings.transformer_model_name)

## Tokens, input IDs, and attention masks

Input IDs index the tokenizer vocabulary. Attention-mask ones mark real tokens; zeros mark padding. Padding is demonstrated here, but training uses dynamic padding so each batch pads only to its own longest sequence.

In [ ]:
examples = splits['train'][settings.text_column].head(2).tolist()
encoded = tokenizer(examples, padding=True, truncation=True, return_tensors='pt')
print('Text:', examples[0])
print('Tokens:', tokenizer.convert_ids_to_tokens(encoded['input_ids'][0]))
print('Input IDs:', encoded['input_ids'][0].tolist())
print('Attention mask:', encoded['attention_mask'][0].tolist())

## Data-driven maximum length

The policy takes the 95th percentile of token lengths in training data only, with a 32-token minimum and 256-token cap. This avoids test-set leakage and avoids blindly padding short support messages to 512.

In [ ]:
from src.train_transformer import choose_max_length
max_length, length_stats = choose_max_length(tokenizer, splits['train'][settings.text_column].tolist())
length_stats

## Fine-tune, evaluate, and save

The classification head outputs one logit per intent. Fine-tuning updates both that head and the pretrained encoder. Validation macro F1 selects the best epoch; test data is evaluated only after training.

In [ ]:
from src.train_transformer import train
metrics = train()
metrics

## Analyze rather than celebrate one score

Inspect macro and weighted F1, the per-class CSV, confusion matrix, confident errors, and `model_comparison.csv`. Training and inference timing are hardware-dependent. Do not change the system from repeated test-set inspection; use validation data for further tuning.